# 07 · Revision analysis: corrected statistics, controls, sensitivity checks

**Settings:** Accelerator *None*, Internet on.
**Inputs:** `bbd-code` (revised) and the outputs of notebooks **01**, **02**, **03**, **05** and **06**.

* Hypothesis table with small-sample t inference, between-cell SD and 95% prediction intervals; H2a paired within
  baseline runs; every row labelled primary, pre-specified follow-up or exploratory.
* Reachability predicted *before* the test set is seen (nominal s; validation-set range).
* H1 decomposition (saturated against unsaturated inputs) and the unbounded head without output bias.
* Q against Q-noent at every scale; classical readout controls; learning-rate sweep; capped validation;
  near-duplicate (leakage) audit; benign false-alarm rates at the natural prior; selected hyperparameters.
* All main figures redrawn with the corrected code, plus the revision figures. Writes `revision_assets.zip`.

In [ ]:
# Pinned: these are the versions the hardware code was tested with (SamplerV2 API).
%pip install -q "qiskit==2.6.1" "qiskit-ibm-runtime==0.51.0" "qiskit-aer==0.18.0" pylatexenc

In [ ]:
# --- bootstrap: find the bbd-code dataset, copy it to a writable folder, import it
import glob, os, shutil, sys
# only the bbd-code dataset counts: attached notebook outputs never contain the code (it is copied to /tmp)
hits = [h for h in sorted(glob.glob("/kaggle/input/**/bbd/__init__.py", recursive=True)) if "/bbd_code/" not in h]
hits = hits or sorted(glob.glob("./bbd/__init__.py"))
if not hits:
    raise FileNotFoundError("Attach the 'bbd-code' dataset to this notebook (KAGGLE_INSTRUCTIONS.md, step 2).")
if len(hits) > 1:
    print("warning: several copies of the code are attached; using", hits[0])
SRC = os.path.dirname(os.path.dirname(os.path.abspath(hits[0])))
CODE = "/tmp/bbd_code" if os.path.isdir("/kaggle/working") else SRC
if CODE != SRC:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(SRC, CODE, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "results", "work"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
import bbd
print("bbd", bbd.__version__, "loaded from", SRC)

In [ ]:
import json
from pathlib import Path
import pandas as pd
from bbd import config, figures, revision, runner, stats, tables
RESULTS = config.RESULTS_DIR
runner.carry_forward(RESULTS, config.INPUT_ROOT, kinds=("ledger",))
hw_out = None
for f in sorted(Path(config.INPUT_ROOT).glob("**/hardware/hw_results.json")):
    hw_out = json.loads(f.read_text()); print("hardware session 1:", f)
df = stats.records_frame(RESULTS)
print(len(df), "records,", df["run_id"].nunique(), "runs")

## Hypotheses (corrected inference)

In [ ]:
hyp = stats.hypothesis_table(df)
pd.set_option("display.width", 220)
display(hyp.reindex(columns=["hypothesis", "role", "claim", "test", "method", "n", "n_cells", "est", "ci90", "ci95",
                             "pi95", "tau", "p", "p_holm"]))
tables.write_all(df, hyp, config.TABLE_DIR)
hyp.to_csv(config.TABLE_DIR / "table_hypotheses_full.csv", index=False)
base = df[(df.tags_str == "") & (df.rho_str != "nat") & df.dataset.isin(["toniot8", "ciciot8"])]
cells = stats.cell_table(stats.paired(base, {"model": "Q", "variant": "raw", "loss": ["CE", "LA"], "s_label": stats.FIXED_S},
                                      {"model": "Q-noent", "variant": "raw"}))
cells.to_csv(config.TABLE_DIR / "rev_q_vs_qnoent_cells.csv", index=False)
print("Q - Q-noent cells inside / outside the 1-point margin:", int(cells.inside_margin.sum()), "/",
      int(cells.outside_margin.sum()), "of", len(cells))

## Revision analyses

In [ ]:
rev = revision.run_all(df, RESULTS, config.TABLE_DIR, config.FIG_DIR)
for name in ("reachability_pretest", "h1_decomposition", "h1_nobias", "q_vs_qnoent", "lr_sweep", "valcap", "leakage"):
    v = rev.get(name)
    print("=" * 20, name)
    for frame in (v if isinstance(v, tuple) else (v,)):
        if isinstance(frame, pd.DataFrame) and len(frame):
            display(frame.round(3))

In [ ]:
ctl = rev.get("controls")
if isinstance(ctl, pd.DataFrame) and len(ctl):
    display(ctl[ctl.s == "all"].round(2))
npt = rev.get("natural_prior")
if isinstance(npt, pd.DataFrame) and len(npt):
    display(npt.round(3))

## Hardware session 2

In [ ]:
for f in sorted(Path(config.INPUT_ROOT).glob("**/hardware/session2/hw2_tests.json")):
    print(f); print(f.read_text()[:4000])
for f in sorted(Path(config.INPUT_ROOT).glob("**/hardware/session2/hw_results.json")):
    h2 = json.loads(f.read_text())
    display(pd.DataFrame({nm: {c: round(v["macro_f1"], 3) for c, v in cs.items()} for nm, cs in h2["conditions"].items()}).T)

## Figures (main figures redrawn with the corrected code, plus revision figures)

In [ ]:
shot_df = figures.shot_curve(RESULTS, df)
made = figures.make_all(df, config.FIG_DIR, RESULTS, hw_out, shot_df)
from IPython.display import Image, display
for p in sorted(config.FIG_DIR.glob("*.png")):
    print(p.name); display(Image(str(p), width=860))

## Paper figures (Figs. 2, 4 and 5 of the manuscript)

Drawn only from released outputs: the main-grid ledger (notebook 02), `rev_h1_decomposition.csv` (above) and the two
hardware records (notebooks 03 and 06). Fig. 3 is `fig2_main` above; Fig. 1 is a TikZ schematic in `main.tex`.

In [ ]:
from bbd import paper_figures
PAPER_FIG = config.WORK_ROOT / "paper_figures"
hw1 = sorted(Path(config.INPUT_ROOT).glob("**/hardware/hw_results.json"))
hw2 = sorted(Path(config.INPUT_ROOT).glob("**/hardware/session2/hw_results.json"))
if hw1 and hw2:
    made, reach = paper_figures.make_all(df, config.TABLE_DIR, hw1[0], hw2[0], PAPER_FIG)
    for p in made:
        if p.suffix == ".png":
            print(p.name); display(Image(str(p), width=860))
else:
    print("attach the outputs of notebooks 03 and 06 to draw the hardware figure")

## Paper numbers not printed above

Recomputes every number in the manuscript that no other cell prints (derived from the ledgers, the hardware records
and two small state-vector checks) and sets it beside the value the paper states. `agrees` is `yes` when they match
at the paper's rounding. Needs the outputs of notebooks 01, 02, 03, 05 and 06; no training, CPU only.

In [ ]:
from bbd import paper_numbers
find = lambda pat: (sorted(Path(config.INPUT_ROOT).glob(pat)) or [None])[0]
numbers = paper_numbers.paper_numbers(
    df, RESULTS, config.TABLE_DIR,
    hw1=find("**/hardware/hw_results.json"), hw2=find("**/hardware/session2/hw_results.json"),
    hw2_tests=find("**/hardware/session2/hw2_tests.json"), dataset_summary=find("**/prepared/dataset_summary.csv"))
PAPER_FIG.mkdir(parents=True, exist_ok=True)
numbers.to_csv(PAPER_FIG / "paper_numbers.csv", index=False)
with pd.option_context("display.max_colwidth", 90, "display.width", 250):
    display(numbers[["section", "claim", "paper", "value", "agrees", "source"]])
print("claims checked:", len(numbers), "| agree:", int((numbers.agrees == "yes").sum()),
      "| differ:", list(numbers.loc[numbers.agrees == "no", "claim"]))

In [ ]:
import zipfile
ZIP = config.WORK_ROOT / "revision_assets.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (config.FIG_DIR, config.TABLE_DIR, config.WORK_ROOT / "paper_figures"):
        for p in (folder.iterdir() if folder.exists() else []):
            z.write(p, f"{folder.name}/{p.name}")
    for f in Path(config.INPUT_ROOT).glob("**/hardware/session2/*.json"):
        z.write(f, f"hardware_session2/{f.name}")
print("wrote", ZIP)